<a href="https://colab.research.google.com/github/gracenaomi1122/my-first-repo/blob/main/Assignment_Data_Integration_and_Querying_Across_Tools_Subjective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import sqlite3
import pandas as pd

# Build an in-memory database from the sample data provided in the question.
conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE genres (genre_id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO genres VALUES (1,'Rock'),(2,'Jazz');

CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO artists VALUES (1,'AC/DC'),(2,'Accept');

CREATE TABLE albums (album_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER);
INSERT INTO albums VALUES (1,'For Those About To Rock',1),(2,'Balls to the Wall',2);

CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, name TEXT, album_id INTEGER, genre_id INTEGER);
INSERT INTO tracks VALUES (1,'For Those About',1,1),(2,'Balls to the Wall',2,1),
                          (6,'Put The Finger On',1,1),(7,'Restless and Wild',2,1);

CREATE TABLE invoice_lines (invoice_line_id INTEGER PRIMARY KEY, invoice_id INTEGER,
                            track_id INTEGER, unit_price REAL, quantity INTEGER);
INSERT INTO invoice_lines VALUES (1,1,2,0.99,1),(2,1,4,0.99,1),(3,2,6,0.99,2),
                                 (4,3,2,0.99,1),(5,3,1,0.99,1);
""")

# --- SQL approach ---
sql_query = """
SELECT
    t.name              AS track_name,
    ar.name             AS artist_name,
    g.name              AS genre_name,
    SUM(il.quantity)    AS total_quantity
FROM invoice_lines AS il
INNER JOIN tracks  AS t  ON il.track_id  = t.track_id
INNER JOIN albums  AS al ON t.album_id   = al.album_id
INNER JOIN artists AS ar ON al.artist_id = ar.artist_id
INNER JOIN genres  AS g  ON t.genre_id   = g.genre_id
GROUP BY t.track_id, t.name, ar.name, g.name
ORDER BY total_quantity DESC;
"""

df_sql = pd.read_sql(sql_query, conn)
print(df_sql.to_string(index=False))

# --- Pandas approach ---
tracks_df        = pd.read_sql("SELECT * FROM tracks",        conn)
albums_df        = pd.read_sql("SELECT * FROM albums",        conn)
artists_df       = pd.read_sql("SELECT * FROM artists",       conn)
genres_df        = pd.read_sql("SELECT * FROM genres",        conn)
invoice_lines_df = pd.read_sql("SELECT * FROM invoice_lines", conn)

def build_report(tracks, albums, artists, genres, invoice_lines):
    # Rename 'name' columns upfront to avoid collision confusion across merges
    tracks  = tracks.rename(columns={'name': 'track_name'})
    artists = artists.rename(columns={'name': 'artist_name'})
    genres  = genres.rename(columns={'name': 'genre_name'})
    # Step 1: inner join invoice_lines with tracks (only purchased tracks kept)
    df = pd.merge(invoice_lines, tracks, on='track_id', how='inner')
    # Step 2: join with albums to bring in artist_id
    df = pd.merge(df, albums, on='album_id', how='inner')
    # Step 3: join with artists to get artist name
    df = pd.merge(df, artists, on='artist_id', how='inner')
    # Step 4: join with genres to get genre name
    df = pd.merge(df, genres, on='genre_id', how='inner')
    # Step 5: aggregate — sum quantity sold per track/artist/genre
    result = (
        df.groupby(['track_name', 'artist_name', 'genre_name'], as_index=False)
          .agg(total_quantity=('quantity', 'sum'))
          .sort_values('total_quantity', ascending=False)
          .reset_index(drop=True)
    )
    return result

report = build_report(tracks_df, albums_df, artists_df, genres_df, invoice_lines_df)
print(report.to_string(index=False))


       track_name artist_name genre_name  total_quantity
Balls to the Wall      Accept       Rock               2
Put The Finger On       AC/DC       Rock               2
  For Those About       AC/DC       Rock               1
       track_name artist_name genre_name  total_quantity
Balls to the Wall      Accept       Rock               2
Put The Finger On       AC/DC       Rock               2
  For Those About       AC/DC       Rock               1
